### Imports

In [1]:
import xarray as xr
import gcsfs
import json
from dask.diagnostics import ProgressBar

from google.cloud import storage
from google.oauth2.credentials import Credentials

import sys
sys.path.append("../src/")
sys.path.append("../ocean_emulators_main/")
from ocean_emulators.dataset_validation import ds_input_validate, ds_prediction_validate
from ocean_emulators.postprocessing import post_processor

## Data to leap

In [2]:
with open(
    "/global/homes/s/suryad/.config/gcloud/application_default_credentials.json"
) as f:  # 🚨 make sure to enter the `.json` file from step 7
    token = json.load(f)
fs = gcsfs.GCSFileSystem(token=token)

#### Define Paths

In [8]:
local_truth_path = "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr" # OM4_5daily_v0.2.1.zarr"
local_pred_path = "/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-08-20_ConvNextUNetTrain3Dv021Eval3DEpochs70Epoch28_Train_global_3D_Test_global_3D_all_N_train_4000_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_4000_rand_seed_1.zarr"
leap_path = "gs://leap-persistent/sd5313/convnext_bestepoch-28_train-OM4v0.2.1_eval-OM4v0.2.1"

In [9]:
ds_truth = xr.open_zarr(local_truth_path)
ds_truth = ds_truth.isel(time=slice(4143, 4743)).isel(lev=slice(None, 19))
ds = xr.open_zarr(local_pred_path)
ds

<xarray.Dataset>
Dimensions:                        (time: 600, x: 180, y: 360, var: 77)
Dimensions without coordinates: time, x, y, var
Data variables:
    __xarray_dataarray_variable__  (time, x, y, var) float64 dask.array<chunksize=(75, 23, 45, 10), meta=np.ndarray>

#### Validation and Postprocessing


In [10]:
# ds_raw_prediction_validate(ds)
ds = post_processor(ds, ds_truth).astype('float32').chunk({'time':10, 'x':-1, 'y':-1, 'lev':-1})
ds_prediction_validate(ds)

/pscratch/sd/s/suryad/Ocean_Emulator/notebooks/../ocean_emulators_main/ocean_emulators/postprocessing.py:20: UserWarning: Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream
  warnings.warn(
/pscratch/sd/s/suryad/Ocean_Emulator/notebooks/../ocean_emulators_main/ocean_emulators/dataset_validation.py:51: UserWarning: This checks nothing yet
  warnings.warn("This checks nothing yet")


In [11]:
ds

<xarray.Dataset>
Dimensions:         (time: 600, y: 180, x: 360, lev: 19, y_b: 181, x_b: 361)
Coordinates:
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
    areacello       (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    lon             (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
    wetmask         (lev, y, x) bool dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * time            (time) object 2014-10-05 12:00:00 ... 2022-12-19 12:00:00
Dimensions without coordinates: y_b, x_b
Data variables:
    uo              (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 19), meta=np.ndarray>
    vo              (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 19), meta=np.ndarray>
    thetao          (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 19), meta=np.ndarray>
    so              (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 19), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(10, 180, 360), meta=np.ndarray>

In [12]:
mapper = fs.get_mapper(leap_path)

with ProgressBar():
    ds.to_zarr(mapper)

[########################################] | 100% Completed | 45.37 s


## Data From Leap

In [5]:
# import an access token
# - option 1: read an access token from a file
with open("token.txt") as f:
    access_token = f.read().strip()

# setup a storage client using credentials
credentials = Credentials(access_token)
fs = gcsfs.GCSFileSystem(token=credentials)

#### Define paths

In [10]:
leap_path = 'gs://leap-persistent/jbusecke/ocean-emulators/OM4_5daily_v0.2.1.zarr'
local_path = '/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr'

In [ ]:
ds = xr.open_dataset(fs.get_mapper('leap_path'), engine='zarr', chunks={})
print(ds)

In [ ]:
# encoding = ds.encoding
# for v in ds.variables:
#     if v in encoding.keys():
#         encoding[v]['compressor']=None
#     else:
#         encoding[v] = {'compressor':None}
# with ProgressBar():
#     ds.to_zarr(local_path, encoding = encoding,
#                 consolidated=True, mode='w')


#### Validation

In [11]:
data = xr.open_zarr(local_path)
ds_input_validate(data)